In [ ]:
# 1. Completely remove torchvision to break the circular import loop
!pip uninstall -y torchvision

# 2. Re-install matched, stable audio engine versions
!pip install torch==2.2.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Install WhisperX and a stable transformers build
!pip install whisperx transformers==4.39.3

# 4. Patch supporting audio framework dependencies
!pip install speechbrain==1.0.3 huggingface_hub>=0.23.4

# 5. Ensure system audio utilities are active
!apt-get install ffmpeg -y

!pip install "numpy<2"

Looking in indexes: https://download.pytorch.org/whl/cu121
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
pyannote-metrics 4.1 requires numpy>=2.2.2, but you have numpy 1.26.4 which is incompatible.
pyannote-core 6.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is

In [ ]:
from google.colab import files
import os
import gc
import torch

# 1. Force PyTorch to bypass metadata security check for downloaded model weights
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "true"

import whisperx
import omegaconf

# 2. Upload file directly into the notebook environment
print("--- Step 0: Upload Your Audio File ---")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file was uploaded. Please click 'Choose Files' and select one.")

audio_file_name = list(uploaded.keys())[0]
print(f"Successfully uploaded and targeted: {audio_file_name}\n")

# 3. Pipeline Configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 16 # The 'base' model is lightweight, so 16 works incredibly fast
compute_type = "float16" if device == "cuda" else "int8"

print(f"Using device: {device}")

# 4. Transcribe Audio using the Official 'base' Model
print("\n--- Step 1: Transcribing with Base Model ---")
model = whisperx.load_model("base", device, compute_type=compute_type)
audio = whisperx.load_audio(audio_file_name)

# auto-detect the language
result = model.transcribe(audio, batch_size=batch_size)
detected_language = result["language"]
print(f"Detected Language: {detected_language}")

# Wipe the transcription network from VRAM to clear space for the alignment network
del model
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

# 5. Phonetic Alignment (Calculates exact word boundaries)
print("\n--- Step 2: Aligning Timestamps ---")
model_a, metadata = whisperx.load_align_model(language_code=detected_language, device=device)
final_result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)

del model_a
gc.collect()
if device == "cuda": torch.cuda.empty_cache()

# 6. Print and Save Transcript
print("\n--- Final Aligned Transcript ---")
with open("base_transcript.txt", "w", encoding="utf-8") as f:
    for segment in final_result["segments"]:
        start = segment["start"]
        end = segment["end"]
        text = segment["text"].strip()

        line = f"[{start:.2f}s - {end:.2f}s] {text}"
        print(line)
        f.write(line + "\n")

print("\nProcessing complete! Transcript saved to your files as 'base_transcript.txt'.")

--- Step 0: Upload Your Audio File ---


Saving omnivoice-audio-1781154173519.mp3 to omnivoice-audio-1781154173519 (3).mp3
Successfully uploaded and targeted: omnivoice-audio-1781154173519 (3).mp3

Using device: cuda

--- Step 1: Transcribing with Base Model ---


INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


No language specified, language will be first be detected for each audio file (increases inference time).
Detected language: en (0.78) in first 30s of audio...
Detected Language: en

--- Step 2: Aligning Timestamps ---

--- Final Aligned Transcript ---
[0.38s - 2.02s] Momma Ice Cream Connic Amity.

Processing complete! Transcript saved to your files as 'base_transcript.txt'.


In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 52.9 MB/s eta 0:00:00


In [ ]:
from jiwer import wer, cer

# 1. Reference text
reference = "Mama Ice cream kamma kathi."

# 2. Provide the WhisperX output text
hypothesis = "Momma Ice Cream Connic Amity."

# Calculate Word Error Rate (WER)
error_rate = wer(reference, hypothesis)
accuracy = (1 - error_rate) * 100

# Calculate Character Error Rate (CER)
char_error_rate = cer(reference, hypothesis)
char_accuracy = (1 - char_error_rate) * 100

print("--- Accuracy Metrics ---")
print(f"Word Error Rate (WER): {error_rate:.2%}")
print(f"Word Accuracy: {accuracy:.2f}%\n")

print(f"Character Error Rate (CER): {char_error_rate:.2%}")
print(f"Character Accuracy (Phonetic Precision): {char_accuracy:.2f}%")

--- Accuracy Metrics ---
Word Error Rate (WER): 80.00%
Word Accuracy: 20.00%

Character Error Rate (CER): 51.85%
Character Accuracy (Phonetic Precision): 48.15%
